author: loua

Idea:

1. Split historical elections data by year
2. Use these as first columns: 'Gruppe,ValgstedId,Valgsted navn,KredsNr,Kreds navn,Kommune navn,'
3. Drop columns where all are null/0

## Setup

In [ ]:
import pandas as pd

ROOT = '../../../'
PARENT_PATH = ROOT + 'raw-data/historical-elections/election-results/'
FILENAME = PARENT_PATH + 'ElectionData.csv'
OUTPUT_PATH = ROOT + 'processed-data/elections/'
YEARS = [2009, 2013, 2017, 2021]
MUNICIPALITY_IDS = [101, 147]  # Copenhagen and Frederiksberg
COMMON_COLS = ['Gruppe', 'ValgstedId', 'KredsNr', 'StorKredsNr', 'LandsdelsNr']
PREFIX = 'KV'

def find_polling_areas():
    df_geo = pd.read_csv(PARENT_PATH + 'Geography.csv', sep=';')
    cph_areas = set(df_geo[df_geo['KommuneNr'].isin(MUNICIPALITY_IDS)]['Valgsted Id'].astype(str))
    return cph_areas

CPH_AREAS = find_polling_areas()

In [16]:
df = pd.read_csv(FILENAME, sep=';', dtype=str)
df = df[df['Gruppe'].isin(CPH_AREAS)]
df.info(verbose=True, show_counts=True)

<class 'pandas.core.frame.DataFrame'>
Index: 58 entries, 0 to 57
Data columns (total 305 columns):
 #    Column                           Non-Null Count  Dtype 
---   ------                           --------------  ----- 
 0    Gruppe                           58 non-null     object
 1    ValgstedId                       58 non-null     object
 2    KredsNr                          58 non-null     object
 3    StorKredsNr                      58 non-null     object
 4    LandsdelsNr                      58 non-null     object
 5    KV2009 - Stemmeberettigede       58 non-null     object
 6    KV2009 - Afgivne stemmer         58 non-null     object
 7    KV2009 - Blanke stemmer          58 non-null     object
 8    KV2009 - Andre ugyldige stemmer  58 non-null     object
 9    KV2009 - Gyldige stemmer         58 non-null     object
 10   KV2009 - A                       58 non-null     object
 11   KV2009 - B                       58 non-null     object
 12   KV2009 - C                 

## Convert dtypes to numeric

In [17]:
party_cols = [col for col in df.columns if col.startswith(PREFIX) and any(f'{PREFIX}{year} - ' in col for year in YEARS)]

# To numeric
for col in party_cols:
    df[col] = df[col].replace('-', '')
    df[col] = df[col].str.replace(',', '.', regex=False)
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Drop columns with no non-null values
df = df.dropna(axis=1, how='all')
party_cols = [col for col in df.columns if col.startswith(PREFIX) and any(f'{PREFIX}{year} - ' in col for year in YEARS)]

# Fill NaN with 0 for party columns
for col in party_cols:
    df[col] = df[col].fillna(0)

# Convert to int if no fractional part
for col in party_cols:
    if (df[col].dropna() % 1 == 0).all():
        df[col] = df[col].astype('int64')

df.info(verbose=True, show_counts=True)

<class 'pandas.core.frame.DataFrame'>
Index: 58 entries, 0 to 57
Data columns (total 239 columns):
 #    Column                           Non-Null Count  Dtype  
---   ------                           --------------  -----  
 0    Gruppe                           58 non-null     object 
 1    ValgstedId                       58 non-null     object 
 2    KredsNr                          58 non-null     object 
 3    StorKredsNr                      58 non-null     object 
 4    LandsdelsNr                      58 non-null     object 
 5    KV2009 - Stemmeberettigede       58 non-null     float64
 6    KV2009 - Afgivne stemmer         58 non-null     float64
 7    KV2009 - Blanke stemmer          58 non-null     float64
 8    KV2009 - Andre ugyldige stemmer  58 non-null     float64
 9    KV2009 - Gyldige stemmer         58 non-null     float64
 10   KV2009 - A                       58 non-null     float64
 11   KV2009 - B                       58 non-null     float64
 12   KV2009 - C   

In [18]:
df.head()

,Gruppe,ValgstedId,KredsNr,StorKredsNr,LandsdelsNr,KV2009 - Stemmeberettigede,KV2009 - Afgivne stemmer,KV2009 - Blanke stemmer,KV2009 - Andre ugyldige stemmer,KV2009 - Gyldige stemmer,...,KV2021 - Q,KV2021 - R,KV2021 - V,KV2021 - P,KV2021 - E,KV2021 - Ø,KV2021 - Å,KV2021 - 0,KV2021 - T,KV2021 - Æ
0,101001,101001,1,1,1,11209.98,6883.60,83.98,12.56,6787.05,...,10,16,731,2,2,1532,217,0,24,11
1,101002,101002,1,1,1,6096.00,3507.00,35.00,12.00,3460.00,...,2,13,515,0,4,699,102,0,20,3
2,101003,101003,1,1,1,12272.62,7353.23,65.52,13.12,7274.60,...,6,23,757,1,6,1893,215,0,28,8
3,101005,101005,1,1,1,11459.00,6247.00,57.00,20.00,6170.00,...,7,28,588,0,6,1599,185,0,24,8
4,101006,101006,1,1,1,9866.00,5683.00,72.00,14.00,5597.00,...,5,15,581,3,5,1282,170,2,32,10


## Split by year

In [19]:
year_dfs = {}
for year in YEARS:
    pattern = f'{PREFIX}{year} - '
    year_cols = [col for col in df.columns if col.startswith(pattern)]
    year_df = df[COMMON_COLS + year_cols].copy()
    rename_dict = {col: col.replace(pattern, '') for col in year_cols}
    year_df.rename(columns=rename_dict, inplace=True)
    year_dfs[year] = year_df

df_2009 = year_dfs[2009]
df_2013 = year_dfs[2013]
df_2017 = year_dfs[2017]
df_2021 = year_dfs[2021]

# verify
for year in YEARS:
    print(f"\n{year} shape: {year_dfs[year].shape}")
    print(f"Columns: {len(year_dfs[year].columns)}")


2009 shape: (58, 39)
Columns: 39

2013 shape: (58, 42)
Columns: 42

2017 shape: (58, 36)
Columns: 36

2021 shape: (58, 37)
Columns: 37


In [20]:
df_2021.info()

<class 'pandas.core.frame.DataFrame'>
Index: 58 entries, 0 to 57
Data columns (total 37 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   Gruppe                  58 non-null     object
 1   ValgstedId              58 non-null     object
 2   KredsNr                 58 non-null     object
 3   StorKredsNr             58 non-null     object
 4   LandsdelsNr             58 non-null     object
 5   Stemmeberettigede       58 non-null     int64 
 6   Afgivne stemmer         58 non-null     int64 
 7   Blanke stemmer          58 non-null     int64 
 8   Andre ugyldige stemmer  58 non-null     int64 
 9   Gyldige stemmer         58 non-null     int64 
 10  A                       58 non-null     int64 
 11  B                       58 non-null     int64 
 12  C                       58 non-null     int64 
 13  D                       58 non-null     int64 
 14  F                       58 non-null     int64 
 15  I            

In [21]:
df_2021.info()

<class 'pandas.core.frame.DataFrame'>
Index: 58 entries, 0 to 57
Data columns (total 37 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   Gruppe                  58 non-null     object
 1   ValgstedId              58 non-null     object
 2   KredsNr                 58 non-null     object
 3   StorKredsNr             58 non-null     object
 4   LandsdelsNr             58 non-null     object
 5   Stemmeberettigede       58 non-null     int64 
 6   Afgivne stemmer         58 non-null     int64 
 7   Blanke stemmer          58 non-null     int64 
 8   Andre ugyldige stemmer  58 non-null     int64 
 9   Gyldige stemmer         58 non-null     int64 
 10  A                       58 non-null     int64 
 11  B                       58 non-null     int64 
 12  C                       58 non-null     int64 
 13  D                       58 non-null     int64 
 14  F                       58 non-null     int64 
 15  I            

## Add some geography cols, drop others

In [22]:
def add_geography_columns(df):
    geo_df = pd.read_csv(PARENT_PATH + 'Geography.csv', sep=';', encoding='utf-8', quotechar='"')
    geo_df.columns = geo_df.columns.str.strip().str.strip('"')
    geo_df = geo_df[geo_df['KommuneNr'].astype(str).isin(['101', '147'])].copy()
    
    print("Geography CSV columns:", geo_df.columns.tolist())
    print("First few Valgsted Ids:", geo_df['Valgsted Id'].head().tolist())

    geo_dict = {}
    for _, row in geo_df.iterrows():
        valgsted_id = str(row['Valgsted Id']).strip().strip('"')
        geo_dict[valgsted_id] = {
            'Valgsted navn': str(row['Valgsted navn']).strip().strip('"'),
            'Kreds navn': str(row['Kreds navn']).strip().strip('"'),
            'Kommune navn': str(row['Kommune navn']).strip().strip('"')
        }

    print(f"Total entries in geo_dict: {len(geo_dict)}")
    print(f"Sample keys: {list(geo_dict.keys())[:5]}")
    
    df_result = df.copy()
    valgsted_navne = []
    kreds_navne = []
    kommune_navne = []
    
    for _, row in df_result.iterrows():
        valgsted_ids = str(row['ValgstedId']).split(';')
        v_names = []
        k_names = []
        kom_names = []

        for vid in valgsted_ids:
            vid = vid.strip()
            if vid not in geo_dict:
                print(f"Missing ValgstedId: '{vid}' (length: {len(vid)})")
                print(f"Available keys starting with '101': {[k for k in geo_dict.keys() if k.startswith('101')][:10]}")
                raise ValueError(f"ValgstedId '{vid}' not found in Geography.csv")
            
            v_names.append(geo_dict[vid]['Valgsted navn'])
            k_names.append(geo_dict[vid]['Kreds navn'])
            kom_names.append(geo_dict[vid]['Kommune navn'])

        # Join with semicolons, remove duplicates while preserving order
        valgsted_navne.append(';'.join(v_names))
        
        # For Kreds and Kommune, use single value if all same, otherwise semicolon-separate
        if len(set(k_names)) == 1:
            kreds_navne.append(k_names[0])
        else:
            kreds_navne.append(';'.join(k_names))
            
        if len(set(kom_names)) == 1:
            kommune_navne.append(kom_names[0])
        else:
            kommune_navne.append(';'.join(kom_names))

    df_result['Valgsted navn'] = valgsted_navne
    df_result['Kreds navn'] = kreds_navne
    df_result['Kommune navn'] = kommune_navne
    df_result = df_result.drop(columns=['StorKredsNr', 'LandsdelsNr'])
    
    # Reorder columns
    first_cols = ['Gruppe', 'ValgstedId', 'Valgsted navn', 'KredsNr', 'Kreds navn', 'Kommune navn']
    other_cols = [col for col in df_result.columns if col not in first_cols]
    df_result = df_result[first_cols + other_cols]
    
    return df_result

for year in YEARS:
    year_dfs[year] = add_geography_columns(year_dfs[year])
    print(f"Added geography columns to {year}")

df_2009, df_2013, df_2017, df_2021 = [year_dfs[year] for year in YEARS]

# Check the result
print("\nFirst few rows of df_2021:")
print(df_2021[['Gruppe', 'ValgstedId', 'Valgsted navn', 'KredsNr', 'Kreds navn', 'Kommune navn']].head(10))

Geography CSV columns: ['Valgsted Id', 'KommuneNr', 'Kreds Nr', 'Storkreds Nr', 'Landsdels Nr', 'Valgsted navn', 'Kommune navn', 'Kreds navn', 'Storkreds navn', 'Landsdels navn', 'Valgsted start', 'Valgsted stop']
First few Valgsted Ids: [101001, 101002, 101003, 101005, 101006]
Total entries in geo_dict: 61
Sample keys: ['101001', '101002', '101003', '101005', '101006']
Added geography columns to 2009
Geography CSV columns: ['Valgsted Id', 'KommuneNr', 'Kreds Nr', 'Storkreds Nr', 'Landsdels Nr', 'Valgsted navn', 'Kommune navn', 'Kreds navn', 'Storkreds navn', 'Landsdels navn', 'Valgsted start', 'Valgsted stop']
First few Valgsted Ids: [101001, 101002, 101003, 101005, 101006]
Total entries in geo_dict: 61
Sample keys: ['101001', '101002', '101003', '101005', '101006']
Added geography columns to 2013
Geography CSV columns: ['Valgsted Id', 'KommuneNr', 'Kreds Nr', 'Storkreds Nr', 'Landsdels Nr', 'Valgsted navn', 'Kommune navn', 'Kreds navn', 'Storkreds navn', 'Landsdels navn', 'Valgsted s

In [23]:
df_2021.head()

,Gruppe,ValgstedId,Valgsted navn,KredsNr,Kreds navn,Kommune navn,Stemmeberettigede,Afgivne stemmer,Blanke stemmer,Andre ugyldige stemmer,...,Q,R,V,P,E,Ø,Å,0,T,Æ
0,101001,101001,1. Østerbro,1,1. Østerbro,København,12518,8360,110,25,...,10,16,731,2,2,1532,217,0,24,11
1,101002,101002,1. Nord,1,1. Østerbro,København,6859,4504,50,14,...,2,13,515,0,4,699,102,0,20,3
2,101003,101003,1. Syd,1,1. Østerbro,København,12504,9094,99,23,...,6,23,757,1,6,1893,215,0,28,8
3,101005,101005,1. Vest,1,1. Østerbro,København,12953,7643,117,18,...,7,28,588,0,6,1599,185,0,24,8
4,101006,101006,1. Nordvest,1,1. Østerbro,København,11149,7039,92,29,...,5,15,581,3,5,1282,170,2,32,10


In [24]:
df_2021.info()

<class 'pandas.core.frame.DataFrame'>
Index: 58 entries, 0 to 57
Data columns (total 38 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   Gruppe                  58 non-null     object
 1   ValgstedId              58 non-null     object
 2   Valgsted navn           58 non-null     object
 3   KredsNr                 58 non-null     object
 4   Kreds navn              58 non-null     object
 5   Kommune navn            58 non-null     object
 6   Stemmeberettigede       58 non-null     int64 
 7   Afgivne stemmer         58 non-null     int64 
 8   Blanke stemmer          58 non-null     int64 
 9   Andre ugyldige stemmer  58 non-null     int64 
 10  Gyldige stemmer         58 non-null     int64 
 11  A                       58 non-null     int64 
 12  B                       58 non-null     int64 
 13  C                       58 non-null     int64 
 14  D                       58 non-null     int64 
 15  F            

## Translate Danish columns

In [25]:
DFS = [df_2009, df_2013, df_2017, df_2021]

translations = {
    'ValgstedId': 'PollingAreaID',
    'Valgsted navn': 'Name',
    'KredsNr': 'DistrictNo',
    'Kreds navn': 'District',
    'Kommune navn': 'Municipality',
    'Stemmeberettigede': 'Eligible voters',
    'Afgivne stemmer': 'Voter turnout',
    'Blanke stemmer': 'Blank votes',
    'Andre ugyldige stemmer': 'Other invalid votes',
    'Gyldige stemmer': 'Valid votes'
}

for i, year in enumerate(YEARS):
    DFS[i] = DFS[i].rename(columns=translations)
df_2009, df_2013, df_2017, df_2021 = DFS

In [26]:
df_2021.info()

<class 'pandas.core.frame.DataFrame'>
Index: 58 entries, 0 to 57
Data columns (total 38 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Gruppe               58 non-null     object
 1   PollingAreaID        58 non-null     object
 2   Name                 58 non-null     object
 3   DistrictNo           58 non-null     object
 4   District             58 non-null     object
 5   Municipality         58 non-null     object
 6   Eligible voters      58 non-null     int64 
 7   Voter turnout        58 non-null     int64 
 8   Blank votes          58 non-null     int64 
 9   Other invalid votes  58 non-null     int64 
 10  Valid votes          58 non-null     int64 
 11  A                    58 non-null     int64 
 12  B                    58 non-null     int64 
 13  C                    58 non-null     int64 
 14  D                    58 non-null     int64 
 15  F                    58 non-null     int64 
 16  I              

## Save to CSVs

In [27]:
for i, year in enumerate(YEARS):
    filename = f'{OUTPUT_PATH}absolute_{year}.csv'
    DFS[i].to_csv(filename, index=False)
    print(f"Saved {filename}")

Saved ../../../processed-data/elections/absolute_2009.csv
Saved ../../../processed-data/elections/absolute_2013.csv
Saved ../../../processed-data/elections/absolute_2017.csv
Saved ../../../processed-data/elections/absolute_2021.csv


## Convert to percentages and save to new CSVs

In [28]:
for i, year in enumerate(YEARS):
    df_rel = DFS[i].copy()
    gyldige_idx = df_rel.columns.get_loc('Valid votes')
    party_cols = df_rel.columns[gyldige_idx + 1:].tolist()
    
    # Convert party columns to percentage of 'Gyldige stemmer'
    df_rel[party_cols] = df_rel[party_cols].div(df_rel['Valid votes'], axis=0) * 100

    # Convert 'Blanke stemmer', 'Andre ugyldige stemmer', 'Gyldige stemmer' to percentage of 'Afgivne stemmer'
    for col in ['Blank votes', 'Other invalid votes', 'Valid votes']:
        df_rel[col] = df_rel[col].div(df_rel['Voter turnout'], axis=0) * 100
    
    # Convert 'Afgivne stemmer' to percentage of 'Stemmeberettigede'
    df_rel['Voter turnout'] = df_rel['Voter turnout'].div(df_rel['Eligible voters'], axis=0) * 100
    
    # Save to CSV
    filename = f'{OUTPUT_PATH}relative_{year}.csv'
    df_rel.to_csv(filename, index=False)
    print(f"Saved {filename}")

Saved ../../../processed-data/elections/relative_2009.csv
Saved ../../../processed-data/elections/relative_2013.csv
Saved ../../../processed-data/elections/relative_2017.csv
Saved ../../../processed-data/elections/relative_2021.csv
